In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

In [5]:
# 画像サイズの設定 (MobileNetV2のデフォルト)
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

In [6]:
# 学習データの水増し設定
train_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input, # 正規化
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

In [7]:
# テストデータは正規化のみ
test_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input
)

In [8]:
# データの読み込み
train_generator = train_datagen.flow_from_directory(
    'dog_cat_photos/train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    'dog_cat_photos/test',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

Found 300 images belonging to 2 classes.
Found 100 images belonging to 2 classes.


In [9]:
# ベースモデルの読み込み（重みはImageNetで学習済み、トップ層は除外）
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False  # 特徴抽出部分は固定

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(1, activation='sigmoid') # 二値分類（犬か猫か）
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [11]:
!pip install scipy

   ---------------------------------------- 0.0/38.6 MB ? eta -:--:--
   --- ------------------------------------ 3.1/38.6 MB 15.3 MB/s eta 0:00:03
   --------- ------------------------------ 8.9/38.6 MB 22.1 MB/s eta 0:00:02
   ------------------ --------------------- 17.8/38.6 MB 28.1 MB/s eta 0:00:01
   ---------------------------- ----------- 27.0/38.6 MB 32.3 MB/s eta 0:00:01
   ----------------------------------- ---- 34.6/38.6 MB 32.8 MB/s eta 0:00:01
   ---------------------------------------  38.5/38.6 MB 31.8 MB/s eta 0:00:01
   ---------------------------------------- 38.6/38.6 MB 28.8 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
# データセットが合計400枚と比較的少ないため、過学習に注意しつつエポック学習させる
history = model.fit(
    train_generator,
    epochs=10,
    validation_data=test_generator
)

Epoch 1/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 670ms/step - accuracy: 0.9267 - loss: 0.2095 - val_accuracy: 0.9200 - val_loss: 0.1803
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 652ms/step - accuracy: 0.8967 - loss: 0.2205 - val_accuracy: 0.9200 - val_loss: 0.1971
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 620ms/step - accuracy: 0.9333 - loss: 0.1916 - val_accuracy: 0.9200 - val_loss: 0.1742
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 622ms/step - accuracy: 0.9433 - loss: 0.1667 - val_accuracy: 0.9200 - val_loss: 0.1671
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 611ms/step - accuracy: 0.9533 - loss: 0.1616 - val_accuracy: 0.9200 - val_loss: 0.1706
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 618ms/step - accuracy: 0.9633 - loss: 0.1477 - val_accuracy: 0.9200 - val_loss: 0.1614
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 646ms/step - accuracy: 0.9367 - loss: 0.1803 - val_accuracy: 0.9400 - val_loss: 0.1409
Epoch 8/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 631ms/step - accuracy: 0.9567 - loss: 0.1639 - val_accuracy: 0.

In [14]:
# 最終評価
print("\n[最終評価結果]")

# test_generator（テスト用データ50枚ずつ）を使ってモデルを評価する
loss, accuracy = model.evaluate(test_generator)

print(f'テストデータの正答率（Accuracy）: {accuracy * 100:.2f}%')


[最終評価結果]
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 538ms/step - accuracy: 0.9300 - loss: 0.1472
テストデータの正答率（Accuracy）: 93.00%
